# PDD Mobility Scanner — Cross Slope from Depth Pro

Estimate **cross slope** (lateral tilt) of a walking surface from video using
[Apple Depth Pro](https://github.com/apple/ml-depth-pro).

**Approach:**
1. Extract frames from an uploaded MP4
2. Run Depth Pro to get a metric depth map + focal length per frame
3. Unproject ground-region pixels into 3D points using the depth + focal length
4. Fit a plane to the ground points → decompose into cross slope and running slope
5. Average across frames per segment for noise reduction

**Requirements:** Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU)

## Step 1: Install & load Depth Pro

In [ ]:
# Install depth_pro without its deps to avoid numpy<2 conflict with Colab
!pip install -q --no-deps git+https://github.com/apple/ml-depth-pro.git
!pip install -q timm pillow_heif

import os
os.makedirs("checkpoints", exist_ok=True)
if not os.path.exists("checkpoints/depth_pro.pt"):
    !wget -q --show-progress https://ml-site.cdn-apple.com/models/depth-pro/depth_pro.pt -P checkpoints
    print("Model downloaded.")
else:
    print("Model already present.")

import torch
import depth_pro

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model, transform = depth_pro.create_model_and_transforms(
    device=device,
    precision=torch.float16 if device.type == "cuda" else torch.float32,
)
model.eval()
print(f"Depth Pro loaded on {device}")

## Step 2: Upload video & extract frames

In [ ]:
import cv2
import numpy as np
from PIL import Image
from google.colab import files

uploaded = files.upload()
video_path = list(uploaded.keys())[0]

frame_interval_sec = 1  # @param {type:"number"}

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = total_frames / fps
frame_step = max(1, int(fps * frame_interval_sec))

print(f"Video: {fps:.1f} FPS, {total_frames} frames, {duration:.1f}s")
print(f"Extracting 1 frame every {frame_interval_sec}s")

frames = []
frame_indices = []
idx = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    if idx % frame_step == 0:
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        frame_indices.append(idx)
    idx += 1
cap.release()
print(f"Extracted {len(frames)} frames.")

## Step 3: Run Depth Pro on each frame

In [ ]:
import time

depth_maps = []   # in mm
focal_lengths = []  # in pixels

for i, rgb in enumerate(frames):
    t0 = time.time()
    pil_img = Image.fromarray(rgb)
    img_tensor = transform(pil_img).to(device)

    with torch.no_grad():
        prediction = model.infer(img_tensor)

    depth = prediction["depth"].cpu().numpy() * 1000  # meters → mm
    f_px = prediction["focallength_px"].cpu().item()
    depth_maps.append(depth)
    focal_lengths.append(f_px)

    elapsed = time.time() - t0
    timestamp = frame_indices[i] / fps
    print(f"  Frame {i+1}/{len(frames)} (t={timestamp:.1f}s) — "
          f"depth: {depth.min():.0f}–{depth.max():.0f} mm, "
          f"f={f_px:.0f}px, {elapsed:.2f}s")

print(f"\nDone. Processed {len(depth_maps)} frames.")

## Step 4: Configure ground region

Select which part of the image is "ground". By default, the bottom 20% of the frame.
Preview the selected region on the first frame to verify.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

ground_top_pct = 80   # @param {type:"slider", min:50, max:95, step:5}
ground_bot_pct = 100  # @param {type:"slider", min:55, max:100, step:5}

h, w = depth_maps[0].shape
row_top = int(h * ground_top_pct / 100)
row_bot = int(h * ground_bot_pct / 100)

print(f"Depth map size: {w}×{h}")
print(f"Ground band: rows {row_top}–{row_bot} ({row_bot - row_top} rows)")

# Preview on first frame
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].imshow(frames[0])
# Scale the rectangle to the original image size
img_h, img_w = frames[0].shape[:2]
rect_top = int(img_h * ground_top_pct / 100)
rect_bot = int(img_h * ground_bot_pct / 100)
rect = patches.Rectangle((0, rect_top), img_w, rect_bot - rect_top,
                          linewidth=2, edgecolor='lime', facecolor='lime', alpha=0.3)
axes[0].add_patch(rect)
axes[0].set_title("Ground region (green)")
axes[0].axis("off")

axes[1].imshow(depth_maps[0], cmap="turbo")
rect2 = patches.Rectangle((0, row_top), w, row_bot - row_top,
                           linewidth=2, edgecolor='lime', facecolor='none')
axes[1].add_patch(rect2)
axes[1].set_title("Depth map — ground band outlined")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## Step 5: Compute cross slope per frame

For each frame:
1. Unproject ground-band pixels to 3D points using depth + focal length
2. Fit a plane to those 3D points via least squares
3. Decompose the plane normal into **cross slope** (lateral tilt) and **running slope** (forward tilt)

Camera coordinate system: X = right, Y = down, Z = forward (depth).
- Cross slope = tilt about the Z axis → `dY/dX` → lateral grade
- Running slope = tilt about the X axis → `dY/dZ` → forward grade

In [ ]:
def compute_slopes(depth_mm, f_px, row_top, row_bot):
    """Fit a plane to ground-band 3D points, return cross & running slope."""
    h, w = depth_mm.shape
    cx, cy = w / 2.0, h / 2.0

    # Pixel coordinates for the ground band
    vs, us = np.mgrid[row_top:row_bot, 0:w]
    depths = depth_mm[row_top:row_bot, :]

    # Filter out invalid depths
    valid = depths > 0
    us_v = us[valid].astype(np.float64)
    vs_v = vs[valid].astype(np.float64)
    ds_v = depths[valid].astype(np.float64)

    if len(ds_v) < 100:
        return np.nan, np.nan, 0

    # Unproject to 3D (camera coords: X=right, Y=down, Z=forward)
    X = (us_v - cx) * ds_v / f_px
    Y = (vs_v - cy) * ds_v / f_px
    Z = ds_v

    # Fit plane: Y = a*X + b*Z + c  (least squares)
    # a = cross slope (dY/dX), b = running slope (dY/dZ)
    A = np.column_stack([X, Z, np.ones(len(X))])
    result = np.linalg.lstsq(A, Y, rcond=None)
    coeffs = result[0]

    cross_slope = coeffs[0]    # dY/dX — lateral tilt
    running_slope = coeffs[1]  # dY/dZ — forward tilt

    return cross_slope, running_slope, len(ds_v)


results = []
for i in range(len(depth_maps)):
    cs, rs, n_pts = compute_slopes(depth_maps[i], focal_lengths[i], row_top, row_bot)
    timestamp = frame_indices[i] / fps
    results.append({
        "frame": i,
        "timestamp_s": timestamp,
        "cross_slope_pct": cs * 100,     # as percentage
        "running_slope_pct": rs * 100,
        "cross_slope_deg": np.degrees(np.arctan(cs)),
        "running_slope_deg": np.degrees(np.arctan(rs)),
        "ground_points": n_pts,
    })
    print(f"  Frame {i+1}: cross={cs*100:+.2f}%  running={rs*100:+.2f}%  "
          f"({np.degrees(np.arctan(cs)):+.2f}° / {np.degrees(np.arctan(rs)):+.2f}°)  "
          f"[{n_pts} pts]")

print(f"\nADA max cross slope: 2.0% (1.15°)")

## Step 6: Segment-averaged cross slope

Average across a sliding window of frames to reduce per-frame noise.

In [ ]:
import pandas as pd

df = pd.DataFrame(results)

window_sec = 5  # @param {type:"number"}
window_frames = max(1, int(window_sec / frame_interval_sec))

df["cross_slope_avg_pct"] = df["cross_slope_pct"].rolling(
    window=window_frames, center=True, min_periods=1
).mean()
df["running_slope_avg_pct"] = df["running_slope_pct"].rolling(
    window=window_frames, center=True, min_periods=1
).mean()

print(f"Averaging over {window_frames}-frame windows ({window_sec}s)")
print(f"\nOverall cross slope:  mean={df['cross_slope_pct'].mean():+.2f}%  "
      f"std={df['cross_slope_pct'].std():.2f}%")
print(f"Overall running slope: mean={df['running_slope_pct'].mean():+.2f}%  "
      f"std={df['running_slope_pct'].std():.2f}%")

ada_violations = (df["cross_slope_avg_pct"].abs() > 2.0).sum()
print(f"\nSegments exceeding ADA 2% cross slope limit: "
      f"{ada_violations}/{len(df)}")

df

## Step 7: Visualize

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Cross slope
axes[0].plot(df["timestamp_s"], df["cross_slope_pct"],
             "o", alpha=0.4, markersize=4, label="Per-frame")
axes[0].plot(df["timestamp_s"], df["cross_slope_avg_pct"],
             "-", linewidth=2, label=f"{window_sec}s average")
axes[0].axhline(y=2.0, color="red", linestyle="--", alpha=0.7, label="ADA limit (±2%)")
axes[0].axhline(y=-2.0, color="red", linestyle="--", alpha=0.7)
axes[0].axhline(y=0, color="gray", linestyle=":", alpha=0.5)
axes[0].set_ylabel("Cross slope (%)")
axes[0].set_title("Cross slope — lateral tilt of walking surface")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Running slope
axes[1].plot(df["timestamp_s"], df["running_slope_pct"],
             "o", alpha=0.4, markersize=4, label="Per-frame")
axes[1].plot(df["timestamp_s"], df["running_slope_avg_pct"],
             "-", linewidth=2, label=f"{window_sec}s average")
axes[1].axhline(y=5.0, color="red", linestyle="--", alpha=0.7, label="ADA limit (5%)")
axes[1].axhline(y=-5.0, color="red", linestyle="--", alpha=0.7)
axes[1].axhline(y=0, color="gray", linestyle=":", alpha=0.5)
axes[1].set_ylabel("Running slope (%)")
axes[1].set_xlabel("Time (s)")
axes[1].set_title("Running slope — forward grade of walking surface")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 8: Visualize ground plane fit on sample frames

In [ ]:
# Show 4 evenly spaced frames with depth cross-section
n_show = min(4, len(frames))
show_idx = np.linspace(0, len(frames) - 1, n_show, dtype=int)

fig, axes = plt.subplots(n_show, 3, figsize=(16, 4 * n_show))
if n_show == 1:
    axes = axes[np.newaxis, :]

for row, fi in enumerate(show_idx):
    depth = depth_maps[fi]
    f_px = focal_lengths[fi]
    h_d, w_d = depth.shape
    mid_row = (row_top + row_bot) // 2
    timestamp = frame_indices[fi] / fps

    # Original frame with ground band
    axes[row, 0].imshow(frames[fi])
    img_h = frames[fi].shape[0]
    r_top = int(img_h * ground_top_pct / 100)
    r_bot = int(img_h * ground_bot_pct / 100)
    rect = patches.Rectangle((0, r_top), frames[fi].shape[1], r_bot - r_top,
                              linewidth=2, edgecolor='lime', facecolor='lime', alpha=0.25)
    axes[row, 0].add_patch(rect)
    axes[row, 0].set_title(f"t={timestamp:.1f}s")
    axes[row, 0].axis("off")

    # Depth map
    axes[row, 1].imshow(depth, cmap="turbo")
    axes[row, 1].axhline(y=mid_row, color="white", linewidth=1, linestyle="--")
    axes[row, 1].set_title(f"Depth map")
    axes[row, 1].axis("off")

    # Cross-section: depth profile along the middle row of ground band
    profile = depth[mid_row, :]
    x_mm = (np.arange(w_d) - w_d / 2.0) * profile / f_px  # lateral position in mm
    y_mm = (mid_row - h_d / 2.0) * profile / f_px  # height in mm

    axes[row, 2].plot(x_mm, y_mm, linewidth=1.5)
    axes[row, 2].set_xlabel("Lateral position (mm)")
    axes[row, 2].set_ylabel("Height (mm)")
    cs_val = df.loc[fi, "cross_slope_pct"]
    axes[row, 2].set_title(f"Ground cross-section — cross slope: {cs_val:+.2f}%")
    axes[row, 2].grid(True, alpha=0.3)
    # Equal aspect so slope is visually accurate
    axes[row, 2].set_aspect("equal")

plt.tight_layout()
plt.show()

## Step 9: Export results

In [ ]:
output_cols = ["frame", "timestamp_s",
               "cross_slope_pct", "cross_slope_avg_pct", "cross_slope_deg",
               "running_slope_pct", "running_slope_avg_pct", "running_slope_deg",
               "ground_points"]

df[output_cols].to_csv("cross_slope_results.csv", index=False)
files.download("cross_slope_results.csv")
print(f"Exported {len(df)} frames to cross_slope_results.csv")
print(f"\nSummary:")
print(f"  Cross slope:   {df['cross_slope_avg_pct'].mean():+.2f}% mean, "
      f"{df['cross_slope_avg_pct'].abs().max():.2f}% peak")
print(f"  Running slope: {df['running_slope_avg_pct'].mean():+.2f}% mean, "
      f"{df['running_slope_avg_pct'].abs().max():.2f}% peak")
print(f"  ADA cross slope violations (>2%): {ada_violations}/{len(df)} segments")